# Complete Qwen 3B reranker experiment

Run the cells in order. This notebook runs all five techniques and stores everything separately under `results/reranker_3b_all` and Google Drive `farmer-llm-reranker-3b`. The 60-question evaluation is raw generation only; judging is a separate step.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select a T4 GPU in Runtime > Change runtime type'
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, json, shutil
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive/farmer-llm-reranker-3b')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
REPO=Path('/content/farmer-llm-project')
URL='https://github.com/sarthakmehra02/stuff.git'
if not REPO.exists(): subprocess.run(['git','clone',URL,str(REPO)],check=True)
else: subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
os.chdir(REPO)
os.environ.pop('FARMER_MODEL_ID',None)
print('Using Qwen 3B')

In [ ]:
%pip install -r requirements-colab.txt
from config import MODEL_ID
print('Model:', MODEL_ID)
assert MODEL_ID == 'Qwen/Qwen2.5-3B-Instruct'

In [ ]:
from src.rag import build_index
index,chunks=build_index()
print('Index:',index.ntotal,'chunks:',len(chunks))

In [ ]:
%%time
!python src/run_conditions.py --conditions zero_shot,prompted,rag --limit 5 --out results/reranker_3b_all --rerank --candidate-k 20
for c in ['zero_shot','prompted','rag']:
    print('\n###',c)
    for line in open(f'results/reranker_3b_all/raw_{c}.jsonl',encoding='utf-8'):
        row=json.loads(line); print(row['id'],row['answer'])

The preview above is complete. The next cell runs the full three non-fine-tuned conditions.

In [ ]:
%%time
!python src/run_conditions.py --conditions zero_shot,prompted,rag --out results/reranker_3b_all --rerank --candidate-k 20 --resume

In [ ]:
!python src/split_train.py --input data/train_qa.jsonl --out-dir data/train_split --val-fraction 0.15

In [ ]:
%%time
!python src/finetune.py --train data/train_split/train.jsonl --val data/train_split/val.jsonl --out /content/drive/MyDrive/farmer-llm-reranker-3b/adapter_run --epochs 3 --max-seq-length 1024 --learning-rate 1e-4

In [ ]:
%%time
!python src/run_conditions.py --conditions finetuned,finetuned_rag --limit 5 --adapter /content/drive/MyDrive/farmer-llm-reranker-3b/adapter_run/adapter --out results/reranker_3b_all --rerank --candidate-k 20
for c in ['finetuned','finetuned_rag']:
    print('\n###',c)
    for line in open(f'results/reranker_3b_all/raw_{c}.jsonl',encoding='utf-8'):
        row=json.loads(line); print(row['id'],row['answer'])

The fine-tuned preview is complete. The next cell runs the final two techniques for all 60 questions.

In [ ]:
%%time
!python src/run_conditions.py --conditions finetuned,finetuned_rag --adapter /content/drive/MyDrive/farmer-llm-reranker-3b/adapter_run/adapter --out results/reranker_3b_all --rerank --candidate-k 20 --resume
for p in Path('results/reranker_3b_all').glob('raw_*.jsonl'): shutil.copy2(p,DRIVE_ROOT/p.name)
shutil.copy2('results/reranker_3b_all/run_metadata.json',DRIVE_ROOT/'run_metadata.json')
print('All 3B outputs copied to',DRIVE_ROOT)